# 🦙 LLaMA 2 Interpretation — Resume Screening Explainer
**Project:** Mitigating Algorithmic Bias in AI-Powered Resume Screening

Pipeline:
1. Load flagged_resumes.csv, shap_proxy_flags.csv, fairness_results.csv
2. Load SHAP token data per resume from shap_results/
3. Build dynamic prompt per resume based on flag type
4. Generate plain-language explanation via LLaMA 2 (local)
5. Save results to llama2_explanations.csv

## 0. Install & Imports

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

import requests

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = Path.cwd()  # Current working directory
FLAGS_CSV       = BASE_DIR / 'flagged_resumes.csv'
SHAP_PROXY_CSV  = BASE_DIR / 'shap_proxy_flags.csv'
FAIRNESS_CSV    = BASE_DIR / 'fairness_results.csv'
SHAP_DIR        = BASE_DIR / 'shap_results'
OUTPUT_CSV      = BASE_DIR / 'llama2_explanations.csv'

# ── Config ────────────────────────────────────────────────────────────────────
PROXY_SHAP_THRESH = 0.005
UNCERTAIN_THRESH  = 0.627
DI_THRESHOLD      = 0.80
TOP_N_TOKENS      = 5      # top positive/negative SHAP tokens to include

print('Config ready ✓')

Config ready ✓


## 1. Load LLaMA 2 Model (4-bit quantized for Colab)

In [2]:
OLLAMA_MODEL = 'llama2'
OLLAMA_API_URL = 'http://localhost:11434/api/generate'

# Test Ollama connection
try:
    response = requests.post(
        OLLAMA_API_URL,
        json={'model': OLLAMA_MODEL, 'prompt': 'test', 'stream': False},
        timeout=5
    )
    print(f'Ollama connected ✓ (Status: {response.status_code})')
except Exception as e:
    print(f'⚠️  Ollama not running. Start it with: ollama serve')
    print(f'Error: {e}')

⚠️  Ollama not running. Start it with: ollama serve
Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=5)


## 2. Load Supporting Data

In [3]:
flags_df   = pd.read_csv(FLAGS_CSV)
proxy_df   = pd.read_csv(SHAP_PROXY_CSV)
metrics_df = pd.read_csv(FAIRNESS_CSV)

# Proxy tokens per occupation (filtered by threshold)
proxy_map = (
    proxy_df[proxy_df['mean_abs'] >= PROXY_SHAP_THRESH]
    .groupby('occupation')['token']
    .apply(list)
    .to_dict()
)

# DI ratio per occupation
di_map = dict(zip(metrics_df['occupation'], metrics_df['di_ratio']))

# Only process flagged resumes
flagged_df = flags_df[flags_df['flags'] != 'NONE'].copy()

print(f'Total flagged resumes : {len(flagged_df)}')
print(f'Proxy map occupations : {len(proxy_map)}')
print('Data loaded ✓')

Total flagged resumes : 283
Proxy map occupations : 9
Data loaded ✓


## 3. SHAP Token Loader

In [4]:
def get_top_shap_tokens(occupation, filename, top_n=TOP_N_TOKENS):
    """
    Load top positive and negative SHAP tokens for a specific resume.
    Returns (top_positive, top_negative) as lists of strings.
    """
    shap_path = SHAP_DIR / f'shap_{occupation}.json'

    if not shap_path.exists():
        return [], []

    with open(shap_path) as f:
        shap_data = json.load(f)

    # Find this resume's SHAP record
    resume_shap = next(
        (r for r in shap_data if r['filename'] == filename), None
    )

    if not resume_shap:
        return [], []

    tokens     = resume_shap['tokens']
    shap_vals  = resume_shap['shap_values']

    # Filter noise tokens
    SKIP = {'[cls]', '[sep]', '[pad]', 'the', 'and', 'of', 'to', 'in', 'a', 'an'}
    token_shap = [
        (t, v) for t, v in zip(tokens, shap_vals)
        if isinstance(t, str) and t.lower() not in SKIP and len(t) >= 3
    ]

    top_positive = [
        t for t, v in sorted(token_shap, key=lambda x: x[1], reverse=True)
        if v > 0
    ][:top_n]

    top_negative = [
        t for t, v in sorted(token_shap, key=lambda x: x[1])
        if v < 0
    ][:top_n]

    return top_positive, top_negative

print('get_top_shap_tokens() defined ✓')

get_top_shap_tokens() defined ✓


## 4. Prompt Builder

In [5]:
def build_prompt(row, top_positive, top_negative):
    """
    Build dynamic LLaMA 2 prompt based on flag type(s).
    Handles UNCERTAIN_DECISION, POTENTIAL_BIAS, FAIRNESS_RISK, and combinations.
    """
    filename     = row['filename']
    pred_occ     = row['predicted_occupation']
    score        = row['suitability_score']
    occupation   = row['occupation']

    positive_str = ', '.join(top_positive) if top_positive else 'N/A'
    negative_str = ', '.join(top_negative) if top_negative else 'N/A'

    # Build flag section dynamically
    flag_lines       = []
    flag_instruction = '.'
    n_flags          = row['n_flags']

    if row['uncertain_decision']:
        flag_lines.append(
            f'⚠️ UNCERTAIN DECISION: Suitability score {score:.2f} is below '
            f'confidence threshold {UNCERTAIN_THRESH}. Model prediction is unreliable.'
        )
        flag_instruction = ', and explain why the model confidence is low for this prediction'

    if row['potential_bias']:
        tokens = proxy_map.get(occupation, [])
        tokens_str = ', '.join(tokens) if tokens else 'unknown'
        flag_lines.append(
            f'⚠️ POTENTIAL BIAS: The following proxy variables were detected with '
            f'significant SHAP influence: {tokens_str}. These may reflect demographic '
            f'signals rather than merit-based qualifications.'
        )
        flag_instruction = (
            ', and identify which proxy variables may have inappropriately '
            'influenced this classification decision'
        )

    if row['fairness_risk']:
        di = di_map.get(occupation, 0)
        flag_lines.append(
            f'⚠️ FAIRNESS RISK: Occupation class {occupation} has a Disparate '
            f'Impact Ratio of {di:.2f}, below the {DI_THRESHOLD} threshold (4/5 rule). '
            f'Candidates in this category may be systematically disadvantaged.'
        )
        flag_instruction = (
            ', and note that candidates in this occupation class may be '
            'systematically disadvantaged by the model'
        )

    if n_flags > 1:
        flag_instruction = ', and address all flagged concerns that HR should review'

    flag_section = '\n'.join(flag_lines)

    prompt = f"""[INST] You are an AI assistant helping HR practitioners understand \
resume screening decisions. Be factual, concise, and avoid making assumptions \
beyond what the data shows.

Resume: {filename}
Predicted occupation: {pred_occ}
Suitability score: {score:.2f}/1.00

Key factors that supported this classification:
{positive_str}

Factors that reduced model confidence:
{negative_str}

{flag_section}

In 2-3 sentences, explain why this resume was classified as {pred_occ}{flag_instruction}. [/INST]"""

    return prompt

print('build_prompt() defined ✓')

# Quick test
test_row = flagged_df.iloc[0]
pos, neg = get_top_shap_tokens(test_row['occupation'], test_row['filename'])
print('\nSample prompt:')
print(build_prompt(test_row, pos, neg))

build_prompt() defined ✓

Sample prompt:
[INST] You are an AI assistant helping HR practitioners understand resume screening decisions. Be factual, concise, and avoid making assumptions beyond what the data shows.

Resume: 10041713.txt
Predicted occupation: CONSTRUCTION
Suitability score: 0.60/1.00

Key factors that supported this classification:
est, ima, tor, bas , construction 

Factors that reduced model confidence:
working , with , the , operations , team 

⚠️ UNCERTAIN DECISION: Suitability score 0.60 is below confidence threshold 0.627. Model prediction is unreliable.
⚠️ POTENTIAL BIAS: The following proxy variables were detected with significant SHAP influence: senior. These may reflect demographic signals rather than merit-based qualifications.

In 2-3 sentences, explain why this resume was classified as CONSTRUCTION, and address all flagged concerns that HR should review. [/INST]


## 5. Generate Explanations

In [6]:
results = []
total   = len(flagged_df)

print(f'Generating explanations for {total} flagged resumes...\n')

for i, (_, row) in enumerate(flagged_df.iterrows()):

    # Progress
    if i % 10 == 0:
        print(f'[{i}/{total}] Processing...')

    # Get SHAP tokens
    top_positive, top_negative = get_top_shap_tokens(
        row['occupation'], row['filename']
    )

    # Build prompt
    prompt = build_prompt(row, top_positive, top_negative)

    # Generate explanation via Ollama
    try:
        response = requests.post(
            OLLAMA_API_URL,
            json={
                'model': OLLAMA_MODEL,
                'prompt': prompt,
                'stream': False,
                'temperature': 0.3,
                'top_p': 0.9,
            },
            timeout=60
        )
        response.raise_for_status()
        explanation = response.json().get('response', 'No response').strip()
    except Exception as e:
        explanation = f'Error: {e}'

    results.append({
        'filename':             row['filename'],
        'occupation':           row['occupation'],
        'predicted_occupation': row['predicted_occupation'],
        'suitability_score':    row['suitability_score'],
        'flags':                row['flags'],
        'n_flags':              row['n_flags'],
        'top_positive_tokens':  ', '.join(top_positive),
        'top_negative_tokens':  ', '.join(top_negative),
        'llama2_explanation':   explanation,
    })

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV, index=False)

print(f'\n✓ Done! Saved to {OUTPUT_CSV}')
print(f'Total explanations generated: {len(results_df)}')

Generating explanations for 283 flagged resumes...

[0/283] Processing...
[10/283] Processing...
[20/283] Processing...
[30/283] Processing...
[40/283] Processing...
[50/283] Processing...
[60/283] Processing...
[70/283] Processing...
[80/283] Processing...
[90/283] Processing...
[100/283] Processing...
[110/283] Processing...
[120/283] Processing...
[130/283] Processing...
[140/283] Processing...
[150/283] Processing...
[160/283] Processing...
[170/283] Processing...
[180/283] Processing...
[190/283] Processing...
[200/283] Processing...
[210/283] Processing...
[220/283] Processing...
[230/283] Processing...
[240/283] Processing...
[250/283] Processing...
[260/283] Processing...
[270/283] Processing...
[280/283] Processing...

✓ Done! Saved to c:\Users\miche\dev\RM\llama2\llama2_explanations.csv
Total explanations generated: 283


## 6. Sample Output

In [7]:
# Show sample explanations per flag type
for flag_type in ['UNCERTAIN_DECISION', 'POTENTIAL_BIAS', 'FAIRNESS_RISK']:
    sample = results_df[
        results_df['flags'].str.contains(flag_type, na=False)
    ].head(1)

    if not sample.empty:
        row = sample.iloc[0]
        print(f'\n{"="*60}')
        print(f'FLAG TYPE  : {flag_type}')
        print(f'Resume     : {row["filename"]}')
        print(f'Occupation : {row["predicted_occupation"]} (true: {row["occupation"]})')
        print(f'Score      : {row["suitability_score"]:.2f}')
        print(f'Flags      : {row["flags"]}')
        print(f'\nLLaMA 2 Explanation:')
        print(row['llama2_explanation'])
        print(f'{"="*60}')


FLAG TYPE  : UNCERTAIN_DECISION
Resume     : 10041713.txt
Occupation : CONSTRUCTION (true: CONSTRUCTION)
Score      : 0.60
Flags      : UNCERTAIN_DECISION, POTENTIAL_BIAS

LLaMA 2 Explanation:
Based on the provided resume, the AI model has predicted that the candidate's occupation is likely to be in the Construction industry. The key factors that supported this classification include the mention of various construction-related terms such as "est", "ima", "tor", and "bas" in the resume. However, there are some concerns flagged by the model that may indicate potential biases or demographic signals. Specifically, the term "senior" was detected with significant SHAP influence, which may reflect merit-based qualifications. HR should review these concerns to ensure that the resume screening decision is fair and unbiased.

FLAG TYPE  : POTENTIAL_BIAS
Resume     : 10041713.txt
Occupation : CONSTRUCTION (true: CONSTRUCTION)
Score      : 0.60
Flags      : UNCERTAIN_DECISION, POTENTIAL_BIAS

LLa

## 7. Summary Statistics

In [8]:
print('=== LLaMA 2 Interpretation Summary ===')
print(f'Total explanations : {len(results_df)}')
print(f'\nBy flag type:')
print(f'  UNCERTAIN_DECISION : {results_df["flags"].str.contains("UNCERTAIN_DECISION").sum()}')
print(f'  POTENTIAL_BIAS     : {results_df["flags"].str.contains("POTENTIAL_BIAS").sum()}')
print(f'  FAIRNESS_RISK      : {results_df["flags"].str.contains("FAIRNESS_RISK").sum()}')
print(f'  Multiple flags     : {results_df[results_df["n_flags"] > 1].shape[0]}')
print(f'\nOutput saved to: {OUTPUT_CSV}')
print('\nNext: use llama2_explanations.csv for HR review interface')

=== LLaMA 2 Interpretation Summary ===
Total explanations : 283

By flag type:
  UNCERTAIN_DECISION : 165
  POTENTIAL_BIAS     : 153
  FAIRNESS_RISK      : 72
  Multiple flags     : 95

Output saved to: c:\Users\miche\dev\RM\llama2\llama2_explanations.csv

Next: use llama2_explanations.csv for HR review interface
